# 🎙️ Clone Your Heritage Voice in About an Hour

This notebook demonstrates how to use **VintageVoice** (an F5-TTS fine-tune) to clone a heritage voice on an 8GB GPU.
You provide a reference audio clip (e.g., a family recording) and a text prompt; the model will generate speech that preserves the speaker's identity but applies a historical delivery style.

**Prerequisites:**
- An NVIDIA GPU with ≥8GB VRAM (tested on RTX 3070/4060/5070, T4, V100-8GB).
- Python ≥3.10 installed.
- A reference `.wav` file (5–15 seconds, clean speech, 24 kHz recommended).
- The exact transcript of that reference audio.

---

## Step 1: Install Dependencies

Install `f5-tts` (the inference engine) and `huggingface_hub` for model download.

In [ ]:
!pip install f5-tts huggingface-hub -q

# Verify installation
import importlib
for pkg in ["f5_tts", "huggingface_hub"]:
    try:
        importlib.import_module(pkg)
        print(f"  ✓ {pkg} installed")
    except ImportError:
        print(f"  ✗ {pkg} missing - run the cell again")

## Step 2: Clone this repository (if not already in it)

We need the preset configuration files and helper scripts.

In [ ]:
import os
if not os.path.isdir("vintage-voice"):
    !git clone https://github.com/Scottcjn/vintage-voice.git
    %cd vintage-voice
else:
    print("  ✓ vintage-voice directory exists")
    %cd vintage-voice


## Step 3: Download Model Weights

VintageVoice fine-tuned weights are hosted on Hugging Face.
They will be placed in `./weights/`.

In [ ]:
import os
weights_dir = "./weights"
os.makedirs(weights_dir, exist_ok=True)
if not os.path.exists(os.path.join(weights_dir, "model.safetensors")):
    !huggingface-cli download AutomatedJanitor/vintage-voice model.safetensors vocab.txt --local-dir ./weights
else:
    print("  ✓ Model weights already downloaded")

## Step 4: Prepare Your Reference Audio

Place your `.wav` file somewhere accessible. We'll use a placeholder path below.
**Important:** Provide the exact transcript (`--ref-text`) to avoid reference audio bleed.

Edit the cell below with your actual file path and transcript.

In [ ]:
import os

# === CONFIGURE THESE ===
REF_AUDIO = "path/to/your_voice_5to15_seconds.wav"   # <-- CHANGE THIS
REF_TEXT  = "Exact transcript of your reference clip."  # <-- CHANGE THIS
GEN_TEXT  = "Good evening, I am speaking in a heritage voice."  # what you want to say

# Validate
if not os.path.exists(REF_AUDIO):
    print("⚠ WARNING: REF_AUDIO file not found. Set the correct path above.")
    print("  Place a 5-15 second .wav file and update REF_AUDIO.")
else:
    print(f"  ✓ Reference audio found ({os.path.getsize(REF_AUDIO)} bytes)")

# Export as env vars for shell commands
os.environ["REF_AUDIO"] = REF_AUDIO
os.environ["REF_TEXT"] = REF_TEXT
os.environ["GEN_TEXT"] = GEN_TEXT
print("  ✓ Variables exported")

## Step 5: Generate Voice Clone

We use the `generate.py` script with the `transatlantic` preset as an example.
You can change `--preset` to any of: `transatlantic`, `newsreel`, `fireside`, `radio_drama`, `edison`, `wartime`, `announcer`.
Only `transatlantic` is validated in v0.1.0; others are experimental.

In [ ]:
import os
REF_AUDIO = os.environ.get("REF_AUDIO", "")
REF_TEXT = os.environ.get("REF_TEXT", "")
GEN_TEXT = os.environ.get("GEN_TEXT", "")

if not os.path.exists(REF_AUDIO):
    print("✗ Cannot generate: REF_AUDIO file not found.")
    print("  Go back to Step 4 and set the correct path.")
else:
    print(f"Generating with preset: transatlantic")
    print(f"  Reference: {REF_AUDIO}")
    print(f"  Text: {GEN_TEXT[:50]}...")
    !python scripts/generate.py "$GEN_TEXT" --preset transatlantic --model ./weights/model.safetensors --vocab ./weights/vocab.txt --ref-audio "$REF_AUDIO" --ref-text "$REF_TEXT" --output ./output/cloned_voice.wav
    print("  ✓ Generation complete: ./output/cloned_voice.wav")

## Step 6: Listen to the Result

The generated audio is saved to `./output/cloned_voice.wav`.
You can play it directly in this notebook (if running in Jupyter) or open the file.

In [ ]:
import os, IPython.display as ipd
output_path = "./output/cloned_voice.wav"
if os.path.exists(output_path):
    print(f"Playing: {output_path} ({os.path.getsize(output_path)} bytes)")
    display(ipd.Audio(output_path))
else:
    print("✗ Output file not found. Run Step 5 first.")

## Next Steps
- Experiment with different presets by changing `--preset`.
- Try different reference voices to see how the historical style adapts.
- For fine-tuning on your own heritage language data, see [`scripts/prep_cosyvoice_data.py`](../scripts/prep_cosyvoice_data.py) and the [training pipeline](../scripts/run_pipeline.sh).
- Share your results in the [GitHub Discussions](https://github.com/Scottcjn/vintage-voice/discussions)!

---
*This notebook is part of the [VintageVoice](https://github.com/Scottcjn/vintage-voice) project.
Bounty: [Issue #180](https://github.com/Scottcjn/vintage-voice/issues/180)*